# 04 — LLM: Pollination Report Generation

**Chain:** real CV output (`cv_ml_report.json`) → aggregate → validate → prompt → Gemini → fidelity check → report

This notebook loads the JSON produced by `cv_to_json_report.py` (real CV landings +
a fitted dose-response curve), builds an LLM prompt from it, calls Gemini, and verifies
that every number in the generated report is grounded in the source data (no hallucination).

**HONESTY NOTE:** `cv_facts` fields are real CV measurements. `ml_estimate` fields depend
on an **unverified crop assumption** (see `crop_note` in the JSON) - the prompt forces the
LLM to phrase these as hypothetical ("IF this flower is pomegranate...").

## 1. Setup

In [1]:
import sys, json, logging
from pathlib import Path

sys.path.insert(0, "..")  # run from notebooks/ - adjust if your layout differs
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from src.llm_reporting.metrics import to_facts_dict, validate_bounds
from src.llm_reporting.schemas import ReportInput
from src.llm_reporting.prompts import build_report_prompt
from src.llm_reporting.fidelity_check import check_fidelity
from src.llm_reporting.generate import generate_report

REPORT_PATH = "../docs/schemas/cv_ml_report.json"
print("Using:", REPORT_PATH)

Using: ../docs/schemas/cv_ml_report.json


## 2. Load facts + aggregate

`to_facts_dict()` reads the real CV JSON and attaches an `aggregate` block (totals across
all flowers: pollinator vs non-pollinator visits, species totals, mean/max pollination
score, mean fruit-set probability, total estimated yield).

In [2]:
facts = to_facts_dict(REPORT_PATH)

print("crop_note:", facts["crop_note"])
print()
print(json.dumps(facts["aggregate"], indent=2))

INFO: Computed aggregate: 26 flowers (26 active), 41 pollinator / 11 non-pollinator visits, dominant=['honeybee']


crop_note: These test videos are generic CV-pipeline test footage (Pexels stock videos), NOT verified pomegranate or cucumber orchard recordings. cv_facts are real CV measurements; ml_estimate assumes a crop and must be treated as a hypothesis.

{
  "flower_count": 26,
  "active_flowers": 26,
  "flowers_detected": 26,
  "flowers_analyzed": 26,
  "total_landings": 232,
  "total_real_landings": 52,
  "total_pollinator_visits": 41,
  "total_non_pollinator_visits": 11,
  "total_landing_s": 400.73,
  "species_totals": {
    "honeybee": 19,
    "bee": 4,
    "fly": 14,
    "beetle": 10,
    "bug": 1,
    "butterfly": 4
  },
  "dominant_pollinators": [
    "honeybee"
  ],
  "dominant_pollinator_visits": 19,
  "mean_pollination_score": 63.87,
  "max_pollination_score": 472.0,
  "ml_estimate_summary": {
    "assumed_crop_unverified": true,
    "mean_fruit_set_probability": 0.5031,
    "mean_fruit_set_probability_pct": 50.3,
    "total_estimated_yield_kg": 3.9237,
    "flowers_with_estimate": 26

## 3. Validate bounds

Catches logically impossible CV output (e.g. more real landings than total landings) before it ever reaches the LLM.

In [3]:
issues = validate_bounds(facts)
print("Issues found:", len(issues))
for i in issues:
    print(" -", i)

INFO: validate_bounds: all 26 flower(s) passed


Issues found: 0


## 4. Schema validation (Pydantic)

Rejects NaN/Infinity, negative counts, unknown species, and empty structures.

In [4]:
validated = ReportInput(**facts)
print("Schema validation passed.")
print("flower_count:", validated.flower_count)
print("First flower's cv_facts:", validated.flowers[0].cv_facts.flower_id)

Schema validation passed.
flower_count: 26
First flower's cv_facts: flower_1


## 5. Build the prompt

The prompt embeds a few-shot example forcing the LLM to open Section 2 with an explicit pollinator vs non-pollinator comparison sentence - description-only instructions were found to be unreliable for this (see project history).

In [5]:
prompt = build_report_prompt(json.dumps(facts, indent=2))
print(f"Prompt length: {len(prompt)} chars\n")
print(prompt[:1500])
print("...")

Prompt length: 91739 chars

You are a factual reporting assistant for a pollination-monitoring
computer-vision system.

STRICT RULES:
1. Use ONLY the facts explicitly present in the provided JSON. Never infer,
   estimate, or invent a value not explicitly present in the input.
2. Never perform arithmetic, unit conversion, percentage calculation, or
   aggregation yourself. Report only the numbers you were given, exactly as given.
3. If a field is missing, null, or not present, do not mention it at all.
4. Never speculate about causes not present in the data.
5. Never suggest a recommendation not directly supported by the given numbers.
6. Tone: concise, factual, farmer-friendly. No marketing language.
7. Maximum length: 300-400 words.
8. cv_facts / aggregate fields are REAL, CV-measured facts - state plainly.
9. ml_estimate / aggregate.ml_estimate_summary fields depend on a crop
   assumption. If assumed_crop_unverified is true, phrase every such number
   as hypothetical - follow the 

## 6. Generate the report (real Gemini call)

Requires `LLM_API_KEY` in a `.env` file at the project root. If you don't have a key handy,
skip to Section 7 for a mocked, API-free demonstration instead.

In [6]:
result = generate_report(REPORT_PATH)

print(result.report_text)
print()
print("Fidelity passed:", result.fidelity_passed)
print("Flagged numbers:", result.flagged_numbers)

INFO: Computed aggregate: 26 flowers (26 active), 41 pollinator / 11 non-pollinator visits, dominant=['honeybee']
INFO: validate_bounds: all 26 flower(s) passed
INFO: Calling Gemini model=gemini-2.5-flash temperature=0.10 max_output_tokens=4096 timeout=30s
INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO: Gemini call succeeded (854 chars returned)
INFO: Fidelity check passed (14 numbers verified)


**1. Summary**
26 flowers were detected and analyzed. All detected flowers were analyzed.

**2. Pollinator Activity**
41 pollinator visits were recorded, compared to 11 non-pollinator visits.
Honeybee was the dominant pollinator, with 19 visits.
Species observed and their total visit counts:
* Honeybee: 19 visits
* Fly: 14 visits
* Beetle: 10 visits
* Bee: 4 visits
* Butterfly: 4 visits
* Bug: 1 visit

**3. Pollination Intensity**
Pollination score is an internal effective-dose metric derived from validated pollinator interactions. The mean pollination score across all flowers was 63.87. The maximum recorded was 472.0.

**4. Hypothetical Pollination & Yield Estimate**
If the observed crop is pomegranate, the model estimates a mean fruit set probability of 50.3% for the 26 analyzed flowers. The corresponding estimated total yield is 3.9237 kg.

Fidelity passed: True
Flagged numbers: []


## 7. (Alternative) Mocked run - no API key needed

Useful for CI/CD or demonstrating the pipeline without spending API credits. The mock
returns a small fake report built only from real aggregate numbers, so fidelity passes;
a second mock deliberately includes a fabricated number ("999") to show fidelity correctly
catching it.

In [7]:
from unittest.mock import patch

agg = facts["aggregate"]
fake_good_report = (
    f"{agg['flower_count']} flowers were monitored. "
    f"{agg['total_pollinator_visits']} pollinator visits were recorded, "
    f"compared to {agg['total_non_pollinator_visits']} non-pollinator visits. "
    f"The mean pollination score was {agg['mean_pollination_score']}."
)

with patch("src.llm_reporting.generate.generate", return_value=fake_good_report):
    mock_result_good = generate_report(REPORT_PATH)

print("--- Good mock ---")
print(mock_result_good.report_text)
print("Fidelity passed:", mock_result_good.fidelity_passed, "| Flagged:", mock_result_good.flagged_numbers)

fake_bad_report = "The system detected 999 unexplained pollinator events."
with patch("src.llm_reporting.generate.generate", return_value=fake_bad_report):
    mock_result_bad = generate_report(REPORT_PATH)

print("\n--- Hallucinated mock (should FAIL) ---")
print(mock_result_bad.report_text)
print("Fidelity passed:", mock_result_bad.fidelity_passed, "| Flagged:", mock_result_bad.flagged_numbers)

INFO: Computed aggregate: 26 flowers (26 active), 41 pollinator / 11 non-pollinator visits, dominant=['honeybee']
INFO: validate_bounds: all 26 flower(s) passed
INFO: Fidelity check passed (4 numbers verified)
INFO: Computed aggregate: 26 flowers (26 active), 41 pollinator / 11 non-pollinator visits, dominant=['honeybee']
INFO: validate_bounds: all 26 flower(s) passed


--- Good mock ---
26 flowers were monitored. 41 pollinator visits were recorded, compared to 11 non-pollinator visits. The mean pollination score was 63.87.
Fidelity passed: True | Flagged: []

--- Hallucinated mock (should FAIL) ---
The system detected 999 unexplained pollinator events.
Fidelity passed: False | Flagged: ['999']


--- Good mock ---
26 flowers were monitored. 41 pollinator visits were recorded, compared to 11 non-pollinator visits. The mean pollination score was 63.87.
Fidelity passed: True | Flagged: []

--- Hallucinated mock (should FAIL) ---
The system detected 999 unexplained pollinator events.
Fidelity passed: False | Flagged: ['999']


## 8. Fidelity check in isolation

Demonstrates the standalone checker: decimal/percentage-aware number extraction, a narrow
numeric tolerance (absorbs floating-point noise only, not real rounding), and video-ID
digits excluded from both sides of the comparison.

In [8]:
sample_text = (
    "Mean fruit set probability was 50.31%. "
    "41 pollinator visits vs 11 non-pollinator visits were recorded."
)
passed, flagged = check_fidelity(sample_text, facts)
print("Passed:", passed, "| Flagged:", flagged)

hallucinated_text = "A total of 12345 impossible landings were recorded."
passed2, flagged2 = check_fidelity(hallucinated_text, facts)
print("Passed:", passed2, "| Flagged:", flagged2)

INFO: Fidelity check passed (3 numbers verified)


Passed: True | Flagged: []
Passed: False | Flagged: ['12345']


Passed: True | Flagged: []
Passed: False | Flagged: ['12345']


---
## 9. Summary

| Check | Result |
|---|---|
| Bounds validation | see Section 3 output |
| Schema validation | see Section 4 output |
| Real LLM report fidelity | see Section 6 output |
| Mocked report fidelity (good / hallucinated) | see Section 7 output |

**What this notebook validates:** the full chain (real CV facts → aggregate → validated →
prompted → generated → fidelity-checked) is internally consistent and catches both
malformed input (Section 3-4) and LLM hallucination (Section 7-8).

**What this notebook does NOT validate:** the real-world crop identity of the monitored
flowers (`crop_note` explains this is unverified test footage), or the scientific accuracy
of `ml_estimate` values - those depend on the ML model documented in `03_ml.ipynb`.